# Proyecto empresa aliada: entregable 1

## Cargar las tablas de datos en Pandas:

Utiliza la función de Pandas para cargar cada tabla de datos desde el archivo csv o xlsx (DIM_CATEGORY, DIM_PRODUCT, DIM_SEGMENT, DIM_CALENDAR, FACT_SALES) en un DataFrame.

In [64]:
import pandas as pd
import numpy as np

df_category = pd.read_csv('DIM_CATEGORY.csv')
df_sales = pd.read_csv('FACT_SALES.csv')
df_calendar = pd.read_excel('DIM_CALENDAR.xlsx')
df_product = pd.read_excel('DIM_PRODUCT.xlsx')
df_segment = pd.read_excel('DIM_SEGMENT.xlsx')

##  Revisar y entender los datos cargados:

Realiza una revisión rápida de las primeras filas de cada DataFrame para entender la estructura de los datos y sus principales características. Utiliza métodos como head(), info() y describe() para tener una idea general del contenido de cada tabla.

In [66]:
def revisar_df(df, nombre):
    print("\n" + "\n" + "="*100)
    print(f"REVISIÓN DE: {nombre}")
    print("="*100)

    print(f"\n• Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

    print(f"\n• Nombres de las columnas: {df.columns.tolist()}")
    
    print(f"\n• Tipos de datos por columna: \n{df.dtypes}")

    print(f"\n• Estadísticas numéricas: \n{df.describe()}")

    print(f"\n• Estadísticas de columnas de texto: \n{df.describe(include='object')}")

    print(f"\n• Valores únicos por columna: \n{df.nunique()}")
    
    print(f"\n• Valores nulos por columna: \n{df.isnull().sum()}")

    print(f"\n• Valores nulos totales: {df.isnull().sum().sum()}")

    print(f"\n• Filas duplicadas: {df.duplicated().sum()}")

    if len(df) > 10:
        
        print(f"\n• Primeras 5 filas: \n{df.head()}")

        print(f"\n• Últimas 5 filas: \n{df.tail()}")

        print(f"\n• Muestra de 5 filas aleatorias: \n{df.sample(5)}")

    else:

        print(f"\n• DataFrame completo: \n{df.to_string()}")

revisar_df(df_category, "DIM_CATEGORY")
revisar_df(df_product, "DIM_PRODUCT")
revisar_df(df_segment, "DIM_SEGMENT")
revisar_df(df_calendar, "DIM_CALENDAR")
revisar_df(df_sales, "FACT_SALES")



REVISIÓN DE: DIM_CATEGORY

• Dimensiones: 5 filas x 2 columnas

• Nombres de las columnas: ['ID_CATEGORY', 'CATEGORY']

• Tipos de datos por columna: 
ID_CATEGORY     int64
CATEGORY       object
dtype: object

• Estadísticas numéricas: 
       ID_CATEGORY
count     5.000000
mean      3.000000
std       1.581139
min       1.000000
25%       2.000000
50%       3.000000
75%       4.000000
max       5.000000

• Estadísticas de columnas de texto: 
                              CATEGORY
count                                5
unique                               5
top     FABRIC TREATMENT and SANIT\r\n
freq                                 1

• Valores únicos por columna: 
ID_CATEGORY    5
CATEGORY       5
dtype: int64

• Valores nulos por columna: 
ID_CATEGORY    0
CATEGORY       0
dtype: int64

• Valores nulos totales: 0

• Filas duplicadas: 0

• DataFrame completo: 
   ID_CATEGORY                        CATEGORY
0            1  FABRIC TREATMENT and SANIT\r\n
1            2                

## Realizar la limpieza de datos:

- Corrige posibles inconsistencias en los datos, como errores tipográficos o formatos incorrectos.
- Identifica y maneja valores nulos en cada DataFrame (si es que los hay)
- Identifica si hay duplicados de los DataFrames y en caso de que los haya, eliminalos.

In [68]:
def limpiar_df(df, nombre): # Para  limpiar y estandarizar los DataFrames aplicando varias estrategias
 
    print("\n" + "\n" + "="*100)
    print(f"COMPROBACIÓN DE LIMPIEZA DE: {nombre}")
    print("="*100)
    
# CAPTURAR ESTADO INICIAL
    filas_iniciales = len(df)
    nulos_iniciales = df.isnull().sum().sum()
    dup_iniciales = df.duplicated().sum()
    
    print(f"\nEstado inicial:")
    print(f"  - Filas: {filas_iniciales:,}")
    print(f"  - Valores nulos: {nulos_iniciales}")
    print(f"  - Filas duplicadas: {dup_iniciales}")
    
# ESTANDARIZAR FORMATO DE TEXTO
    for col in df.select_dtypes(include=['object']).columns: # Para columnas de texto (type=object)
        df[col] = df[col].str.strip() # Elimina espacios al inicio/final
        df[col] = df[col].str.title() # Formato Título: Primera Letra Mayúscula
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True) # Reemplaza múltiples espacios con uno solo

# MANEJAR VALORES NULOS (previamente revisados: 14 en DIM_PRODUCT, 1 en DIM_SEGMENT)
    for col in df.select_dtypes(include=['object']).columns: # Para columnas de texto (type=object)
        # ESTRATEGIA 1: ATTR1 - Rellenado basado en dependencia con ATTR2
        if 'ATTR1' in df.columns and 'ATTR2' in df.columns: # Porque ATTR1 tiene dependencia con ATTR2
            for valor_attr2 in df['ATTR2'].unique(): # Para cada valor único de ATTR2
                subset = df[(df['ATTR2'] == valor_attr2) & df['ATTR1'].notna()] # Subconjunto donde ATTR2 = valor_attr2 y ATTR1 no es nulo
                if len(subset) > 0:
                    valor_attr1 = subset['ATTR1'].mode()[0] # Valor más común de ATTR1 para ATTR2
                    coincidencias = (subset['ATTR1'] == valor_attr1).sum() # Calcula % de coincidencia
                    if (coincidencias / len(subset)) * 100 == 100: # Dependencia debe ser 100% para rellenar
                        mascara = (df['ATTR1'].isnull()) & (df['ATTR2'] == valor_attr2)  
                        df.loc[mascara, 'ATTR1'] = valor_attr1
        # ESTRATEGIA 2: ATTR3 - Rellenado con valor fijo                
        if 'ATTR3' in df.columns:                                        
            df['ATTR3'] = df['ATTR3'].fillna('No Definido') # Ya existe valor 'No Definido' en ATTR3
        # ESTRATEGIA 3: ITEM - Rellenado con valor fijo
        if 'ITEM' in df.columns:
            df['ITEM'] = df['ITEM'].fillna('Item Desconocido') # No se conoce el valor del item 
        
# ELIMINAR FILAS DUPLICADAS (si hay después de estandarizar texto y manejar valores nulos se pueden crear filas duplicadas)
    df.drop_duplicates(inplace=True) # Mantiene primera ocurrencia (keep='first') y modifica DataFrame original (inplace=True)
    
# CAPTURAR ESTADO FINAL
    filas_finales = len(df)
    nulos_finales = df.isnull().sum().sum()
    dup_finales = df.duplicated().sum()
    
    # Comparar estado inicial vs final para mostrar impacto de limpieza
    print(f"\nEstado final:")
    print(f"  - Filas: {filas_finales:,}")
    print(f"  - Valores nulos: {nulos_finales}")
    print(f"  - Filas duplicadas: {dup_finales}")
    
    # Mostrar resumen de cambios
    if filas_iniciales != filas_finales or nulos_iniciales != nulos_finales:
        print(f"\nCambios realizados:")
        if nulos_iniciales > nulos_finales:
            print(f"  - Valores nulos corregidos: {nulos_iniciales - nulos_finales}")
        if filas_iniciales > filas_finales:
            print(f"  - Filas duplicadas eliminadas: {filas_iniciales - filas_finales}")
    else:
        print(f"\nDataFrame ya estaba limpio (sin cambios necesarios para valores nulos y duplicados)")
    
    return df

# IMPLEMENTACIÓN DE LIMPIEZA PARA CADA DATAFRAME
df_category = limpiar_df(df_category, "DIM_CATEGORY")
df_product = limpiar_df(df_product, "DIM_PRODUCT")
df_segment = limpiar_df(df_segment, "DIM_SEGMENT")
df_calendar = limpiar_df(df_calendar, "DIM_CALENDAR")
df_sales = limpiar_df(df_sales, "FACT_SALES")

# RESUMEN FINAL DE LIMPIEZA
print("\n" + ":"*100)
print("RESUMEN DE LIMPIEZA")
print(":"*100)
print("\nEstrategias aplicadas:")
print("  1. Estandarización de texto (strip, title, espacios)")
print("  2. Tratamiento inteligente de nulos (dependencias + valores por defecto)")
print("  3. Eliminación de filas duplicadas")

dataframes_limpios = {
    'DIM_CATEGORY': df_category,
    'DIM_PRODUCT': df_product,
    'DIM_SEGMENT': df_segment,
    'DIM_CALENDAR': df_calendar,
    'FACT_SALES': df_sales
}

print(f"\n{'DataFrame':<20} {'Filas':<10} {'Nulos':<10} {'Duplicados':<12}")
print("─"*70)
for nombre, df in dataframes_limpios.items():
    filas = len(df)
    nulos = df.isnull().sum().sum()
    duplicados = df.duplicated().sum()
    print(f"{nombre:<20} {filas:>10,} {nulos:>10} {duplicados:>12}")
print("─"*70)



COMPROBACIÓN DE LIMPIEZA DE: DIM_CATEGORY

Estado inicial:
  - Filas: 5
  - Valores nulos: 0
  - Filas duplicadas: 0

Estado final:
  - Filas: 5
  - Valores nulos: 0
  - Filas duplicadas: 0

DataFrame ya estaba limpio (sin cambios necesarios para valores nulos y duplicados)


COMPROBACIÓN DE LIMPIEZA DE: DIM_PRODUCT

Estado inicial:
  - Filas: 505
  - Valores nulos: 14
  - Filas duplicadas: 0

Estado final:
  - Filas: 505
  - Valores nulos: 0
  - Filas duplicadas: 0

Cambios realizados:
  - Valores nulos corregidos: 14


COMPROBACIÓN DE LIMPIEZA DE: DIM_SEGMENT

Estado inicial:
  - Filas: 53
  - Valores nulos: 1
  - Filas duplicadas: 0

Estado final:
  - Filas: 52
  - Valores nulos: 0
  - Filas duplicadas: 0

Cambios realizados:
  - Valores nulos corregidos: 1
  - Filas duplicadas eliminadas: 1


COMPROBACIÓN DE LIMPIEZA DE: DIM_CALENDAR

Estado inicial:
  - Filas: 156
  - Valores nulos: 0
  - Filas duplicadas: 0

Estado final:
  - Filas: 156
  - Valores nulos: 0
  - Filas duplicadas

## Unir los DataFrames relevantes:

Realiza uniones entre los DataFrames para consolidar la información. Por ejemplo, une la tabla de productos con las ventas, las categorías y los segmentos.

In [70]:
print("="*100)
print("CONSOLIDACIÓN DE DATAFRAMES")
print("="*100)

# Consolidar los 5 DataFrames en uno único
def consolidar_dataframes(df_sales, df_product, df_category, df_segment, df_calendar): 
    
# FACT_SALES + DIM_PRODUCT. Agregar información del producto a cada venta
    print("\nUnión de FACT_SALES con DIM_PRODUCT")
    print("─"*70)
    print(f"Dimensiones de FACT_SALES: {df_sales.shape[0]:,} filas x {df_sales.shape[1]} columnas")
    print(f"Dimensiones de DIM_PRODUCT: {df_product.shape[0]} filas x {df_product.shape[1]} columnas")
    
    # Unir. ITEM_CODE en df_sales = ITEM en df_product
    df_consolidado = pd.merge(
        df_sales, # Tabla izquierda (ventas)
        df_product, # Tabla derecha (dimensión producto)
        left_on='ITEM_CODE', # Llave en SALES
        right_on='ITEM', # Llave en PRODUCT
        how='left', # LEFT JOIN  para mantener todas las ventas
        suffixes=('', '_product') # Agrega sufijo si hay columnas duplicadas 
    )
    print(f"✓ Resultado consolidado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas\n")
    
# + DIM_CATEGORY. Agregar categoría descriptiva de la categorái numérica
    print("\nUnión con DIM_CATEGORY")
    print("─"*70)
    print(f"Dimensiones de DIM_CATEGORY: {df_category.shape[0]} filas x {df_category.shape[1]} columnas")
    
    # Unir. CATEGORY en df_consolidado = ID_CATEGORY en df_category
    df_consolidado = pd.merge(
        df_consolidado, # Resultado del paso anterior
        df_category, # Dimensión categoría
        left_on='CATEGORY', # Columna en df_consolidado (ID numérico)
        right_on='ID_CATEGORY', # Columna en df_category (ID numérico)
        how='left', # LEFT JOIN para mantener todas las filas del paso anterior
        suffixes=('', '_category') # Evita conflicto con columna CATEGORY por duplicación
    )
    print(f"✓ Resultado consolidado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas\n")
    
# + DIM_SEGMENT. Agregar segmentación del producto
    print("\nUnión con DIM_SEGMENT")
    print("─"*70)
    print(f"Dimensiones de DIM_SEGMENT: {df_segment.shape[0]} filas x {df_segment.shape[1]} columnas")
    
    # Unir. Múltiples columnas coinciden en df_consolidado y df_segment: CATEGORY + ATTR1 + ATTR2 + ATTR3 + FORMAT
    df_consolidado = pd.merge(
        df_consolidado, # Resultado del paso anterior
        df_segment, # El segmento depende de la combinación de atributos
        on=['CATEGORY', 'ATTR1', 'ATTR2', 'ATTR3', 'FORMAT'], # Join compuesto
        how='left', # LEFT JOIN porque puede que no todos los productos tengan segmento definido
        suffixes=('', '_segment') # Agrega sufijo si hay columnas duplicadas
    )
    print(f"✓ Resultado consolidado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas\n")
    
# + DIM_CALENDAR. Agregar información temporal
    print("\nUnión con DIM_CALENDAR")
    print("─"*70)
    print(f"Dimensiones de DIM_CALENDAR: {df_calendar.shape[0]} filas x {df_calendar.shape[1]} columnas")
    
    # Unir. WEEK en df_consolidado = WEEK en df_calendar
    df_consolidado = pd.merge(
        df_consolidado, # Resultado del paso anterior
        df_calendar, # Dimensión temporal
        on='WEEK', # Formato semana-año
        how='left', # LEFT JOIN para conservar todas las ventas
        suffixes=('', '_calendar') # Agrega sufijo si hay columnas duplicadas
    )
    print(f"✓ Resultado consolidado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas\n")
    
# VERIFICACIÓN FINAL DE CONSOLIDACIÓN 
    print("\n" + ":"*100)
    print("RESUMEN DE CONSOLIDACIÓN")
    print(":"*100 + "\n")
    
    # Dimensiones finales
    print(f"• Dimensiones del DataFrame consolidado: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas")
    print(f"  ✓ Se mantienen las {len(df_sales):,} ventas originales")
    
    # Lista de columnas
    print(f"\n• Columnas en el DataFrame consolidado:")
    for i, col in enumerate(df_consolidado.columns, 1):
        print(f"{i:2}.{col}")

    # Columnas por origen
    columnas_sales = ['WEEK', 'ITEM_CODE', 'TOTAL_UNIT_SALES', 'TOTAL_VALUE_SALES', 'TOTAL_UNIT_AVG_WEEKLY_SALES', 'REGION']
    columnas_product = ['MANUFACTURER', 'BRAND', 'ITEM', 'ITEM_DESCRIPTION', 'CATEGORY', 'FORMAT', 'ATTR1', 'ATTR2', 'ATTR3']
    columnas_category = ['ID_CATEGORY', 'CATEGORY_category']
    columnas_segment = ['SEGMENT']
    columnas_calendar = ['YEAR', 'MONTH', 'WEEK_NUMBER', 'DATE']

    print(f"\n  De FACT_SALES ({len(columnas_sales)} cols):")
    for col in columnas_sales:
        if col in df_consolidado.columns:
            print(f"    - {col}")
    
    print(f"\n  De DIM_PRODUCT ({len(columnas_product)} cols):")
    for col in columnas_product:
        if col in df_consolidado.columns:
            print(f"    - {col}")
    
    print(f"\n  De DIM_CATEGORY ({len(columnas_category)} cols):")
    for col in columnas_category:
        if col in df_consolidado.columns:
            print(f"    - {col}")
    
    print(f"\n  De DIM_SEGMENT ({len(columnas_segment)} cols):")
    for col in columnas_segment:
        if col in df_consolidado.columns:
            print(f"    - {col}")
    
    print(f"\n  De DIM_CALENDAR ({len(columnas_calendar)} cols):")
    for col in columnas_calendar:
        if col in df_consolidado.columns:
            print(f"    - {col}")

    # Verificar valores nulos
    print(f"\n• Verificación de valores nulos:")
    nulos_totales = df_consolidado.isnull().sum().sum()
    if nulos_totales > 0:
        print(f"  ✖ Hay {nulos_totales:,} valores nulos")
        print("    Nulos por columna:")
        nulos_col = df_consolidado.isnull().sum()
        for col, count in nulos_col[nulos_col > 0].items():
            pct = (count / len(df_consolidado)) * 100
            print(f"    - {col}: {count:,} ({pct:.2f}%)")
    else:
        print("  ✓ No hay valores nulos")
        
    # Verificar duplicados
    print(f"\n• Verificación de valores duplicados:")
    dup_totales = df_consolidado.duplicated().sum()
    if dup_totales > 0:
        print(f"  ✖ Hay {dup_totales:,} filas duplicadas")
    else:
        print("  ✓ No hay filas duplicadas")
    
    # Información del DataFrame
    print(f"\n• Información del DataFrame consolidado:\n")
    df_consolidado.info()
    
    # Muestra aleatoria
    print(f"\n• Muestra de 5 filas aleatorias:\n")
    print(df_consolidado.sample(5))

    return df_consolidado

# IMPLEMENTACIÓN DE CONSOLIDACIÓN DE LOS 5 DATAFRAMES
df_consolidado = consolidar_dataframes(
    df_sales, 
    df_product, 
    df_category, 
    df_segment, 
    df_calendar
)

# ANÁLISIS PRUEBA POST-CONSOLIDACIÓN
print("\n" + "+"*100)
print("ANÁLISIS PRUEBA POST-CONSOLIDACIÓN")
print("+"*100)

print("\nEstadísticas del DataFrame consolidado:")

# Ventas por región
print(f"\n• Ventas por REGIÓN:")
print(df_consolidado.groupby('REGION')['TOTAL_VALUE_SALES'].agg(['count', 'sum', 'mean']).round(2))

# Ventas por segmento
print(f"\n• Ventas por SEGMENTO:")
print(df_consolidado.groupby('SEGMENT')['TOTAL_VALUE_SALES'].agg(['count', 'sum', 'mean']).round(2))

# Ventas por año
print(f"\n• Ventas por AÑO:")
print(df_consolidado.groupby('YEAR')['TOTAL_VALUE_SALES'].agg(['count', 'sum', 'mean']).round(2))

CONSOLIDACIÓN DE DATAFRAMES

Unión de FACT_SALES con DIM_PRODUCT
──────────────────────────────────────────────────────────────────────
Dimensiones de FACT_SALES: 122,002 filas x 6 columnas
Dimensiones de DIM_PRODUCT: 505 filas x 9 columnas
✓ Resultado consolidado: 122,002 filas x 15 columnas


Unión con DIM_CATEGORY
──────────────────────────────────────────────────────────────────────
Dimensiones de DIM_CATEGORY: 5 filas x 2 columnas
✓ Resultado consolidado: 122,002 filas x 17 columnas


Unión con DIM_SEGMENT
──────────────────────────────────────────────────────────────────────
Dimensiones de DIM_SEGMENT: 52 filas x 6 columnas
✓ Resultado consolidado: 122,002 filas x 18 columnas


Unión con DIM_CALENDAR
──────────────────────────────────────────────────────────────────────
Dimensiones de DIM_CALENDAR: 156 filas x 5 columnas
✓ Resultado consolidado: 122,002 filas x 22 columnas


::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
RESUM

## Aplicar transformaciones necesarias:

- Estandariza los formatos de las columnas, como las fechas o categorías, para asegurar la consistencia en todo el conjunto de datos.
- Realiza cualquier transformación adicional requerida para preparar los datos para el análisis, como la creación de nuevas columnas calculadas o la agrupación de datos.

### TRANSFORMACIÓN 1: Eliminar Columnas Duplicadas
### TRANSFORMACIÓN 2: Renombrar Columnas

In [72]:
### TRANSFORMACIÓN 1: Eliminar Columnas Duplicadas

def eliminar_col_dup(df, mantener='primera'): # Detecta y elimina columnas con contenido duplicado (mismo valor en todas las filas)
    
    print("="*100)
    print("TRANSFORMACIÓN 1: Eliminar Columnas Duplicadas")
    print("="*100)

    col_a_eliminar = [] # Lista de columnas a eliminar
    col_procesadas = set() # Evita procesar la misma columna dos veces
    columnas = df.columns.tolist()
    
# DETECTAR COLUMNAS DUPLICADAS
    for i in range(len(columnas)): # Compara cada columna con todas las demás para encontrar duplicados
        if columnas[i] in col_procesadas: # Si ya procesamos esta columna, saltarla
            continue
        col1 = columnas[i]
        col_iguales = [col1] # Lista de columnas con el mismo contenido
        for j in range(i + 1, len(columnas)): # Comparar con el resto de columnas
            col2 = columnas[j]
            if col2 in col_procesadas:
                continue
            if df[col1].equals(df[col2]): #compara todos los valores, fila por fila, retorna True solo si todas las filas son idénticas
                col_iguales.append(col2)
                col_procesadas.add(col2)
                
# DECIDIR CUÁL COLUMNA MANTENER
        if len(col_iguales) > 1: # Si hay duplicados (más de 1 columna igual)
            print(f"\nColumnas duplicadas: {col_iguales}")
            if isinstance(mantener, list): # Si 'mantener' es una lista de preferencias
                mantener_col = None
                for pref in mantener: # Buscar preferencia de columna en la lista
                    if pref in col_iguales:
                        mantener_col = pref
                        break
                if mantener_col is None: # Si no hay preferencia, usar la primera columna
                    mantener_col = col_iguales[0]
                eliminar_cols = [c for c in col_iguales if c != mantener_col] # Columnas a eliminar = todas excepto la que mantenemos
            print(f"   ✓ Mantener: '{mantener_col}' (preferida)")
            print(f"   ✗ Eliminar: {eliminar_cols}")
            col_a_eliminar.extend(eliminar_cols)
    
# ELIMINAR COLUMNAS
    if col_a_eliminar:
        df_limpio = df.drop(columns=col_a_eliminar)
        
        print(f"\nColumnas eliminadas: {len(col_a_eliminar)} ")
        print(f"   - Columnas antes: {len(df.columns)}")
        print(f"   - Columnas después: {len(df_limpio.columns)}")
    else:
        df_limpio = df
        print("\n✓ No hay columnas duplicadas para eliminar")
    return df_limpio

# ESPECIFICAR PREFERENCIA DE COLUMNAS A MANTENER
df_consolidado = eliminar_col_dup(df_consolidado, mantener=['ITEM_CODE', 'ID_CATEGORY'])
    # ITEM_CODE es más descriptivo que ITEM
    # ID_CATEGORY es más claro que CATEGORY (numérico)


### TRANSFORMACIÓN 2: Renombrar Columnas
df_consolidado = df_consolidado.rename(columns={'CATEGORY_category': 'CATEGORY', 'WEEK': 'WEEK_YEAR'}) 
    # CATEGORY_category a CATEGORY (eliminar sufijo redundante)
    # WEEK a WEEK_YEAR (más descriptivo del contenido)
print(f"\nColumnas renombradas: 2")
print(f"   - 'CATEGORY_category': 'CATEGORY'")  
print(f"   - 'WEEK': 'WEEK_YEAR'")

# VERIFICACIÓN FINAL
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 1 Y 2")
print(":"*100)

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas")
print(f"\n• Columnas totales:")
for i, col in enumerate(df_consolidado.columns, 1):
    print(f"{i:2}.{col}")
print(f"\n• Muestra de 5 filas aleatorias:\n {df_consolidado.sample(5)}")

TRANSFORMACIÓN 1: Eliminar Columnas Duplicadas

Columnas duplicadas: ['ITEM_CODE', 'ITEM']
   ✓ Mantener: 'ITEM_CODE' (preferida)
   ✗ Eliminar: ['ITEM']

Columnas duplicadas: ['CATEGORY', 'ID_CATEGORY']
   ✓ Mantener: 'ID_CATEGORY' (preferida)
   ✗ Eliminar: ['CATEGORY']

Columnas eliminadas: 2 
   - Columnas antes: 22
   - Columnas después: 20

Columnas renombradas: 2
   - 'CATEGORY_category': 'CATEGORY'
   - 'WEEK': 'WEEK_YEAR'

::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
RESUMEN DE TRANSFORMACIÓN 1 Y 2
::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::

• Dimensiones totales: 122,002 filas x 20 columnas

• Columnas totales:
 1.WEEK_YEAR
 2.ITEM_CODE
 3.TOTAL_UNIT_SALES
 4.TOTAL_VALUE_SALES
 5.TOTAL_UNIT_AVG_WEEKLY_SALES
 6.REGION
 7.MANUFACTURER
 8.BRAND
 9.ITEM_DESCRIPTION
10.FORMAT
11.ATTR1
12.ATTR2
13.ATTR3
14.ID_CATEGORY
15.CATEGORY
16.SEGMENT
17.YEAR
18.MONTH
19.WEEK_NUMB

### TRANSFORMACIÓN 3: Calcular Columnas Temporales

In [74]:
print("="*100)
print("TRANSFORMACIÓN 3: Calcular Columnas Temporales")
print("="*100)

# Asegurar que DATE sea tipo datetime
df_consolidado['DATE'] = pd.to_datetime(df_consolidado['DATE'], errors='coerce') # Covierte errores a NaT (Not a Time)

print(f"\n  ✓ Tipo de DATE: {df_consolidado['DATE'].dtype}")
print(f"  ✓ Fechas válidas: {df_consolidado['DATE'].notna().sum():,}")
print(f"  ✓ Rango: {df_consolidado['DATE'].min()} a {df_consolidado['DATE'].max()}")

# Extraer componentes de fecha
df_consolidado['MONTH_NAME'] = df_consolidado['DATE'].dt.month_name() # Nombre del mes en inglés

df_consolidado['DAY_NUMBER'] = df_consolidado['DATE'].dt.dayofweek # Día de la semana: 0=Lunes ... 6=Domingo

df_consolidado['DAY_NAME'] = df_consolidado['DATE'].dt.day_name() # Nombre del día en inglés

df_consolidado['QUARTER'] = df_consolidado['DATE'].dt.quarter # Trimestre del año: Q1=Enero-Marzo, Q2=Abril-Junio, Q3=Julio-Sep, Q4=Oct-Dic

df_consolidado['YEAR_MONTH'] = df_consolidado['DATE'].dt.to_period('M').astype(str) # Año-mes en formato texto (YYYY-MM)

df_consolidado['YEAR_QUARTER'] = (df_consolidado['YEAR'].astype(str) + '-Q' + df_consolidado['QUARTER'].astype(str)) # Año-trimestre en formato texto (YYYY-QX)

df_consolidado['MONTH_PERIOD'] = pd.cut( # Periodo del mes por categorías: Inicio Mes=días 1-10, Medio Mes=días 11-20, Fin Mes=días 21-31
    df_consolidado['DATE'].dt.day, # Día del mes (1-31)
    bins=[0, 10, 20, 31], # Cortes: 1-10, 11-20, 21-31
    labels=['Inicio Mes', 'Medio Mes', 'Fin Mes'], # Etiquetas
    include_lowest=True # Incluir el valor más bajo (día 1)
)

# Resumen de columnas de fecha
print(f"\n• Columnas creadas: 7")
print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'MONTH_NAME': 'Agrupar por mes (análisis estacional)',
    'DAY_WEEK': 'Análisis numérico de días (0=Lunes, 6=Domingo)',
    'DAY_NUMBER': 'Visualizaciones legibles de días',
    'QUARTER': 'Análisis trimestral',
    'YEAR_MONTH': 'Series temporales mensuales',
    'YEAR_QUARTER': 'Comparación entre trimestres',
    'MONTH_PERIOD': 'Patrones de compra por quincena/pago'
}
for col, uso in utilidad.items():
    print(f"  - {col:12} → {uso}")

print(f"\n• Muestra de 5 filas aleatorias de fechas: \n{df_consolidado
    [['DATE', 'DAY_NUMBER', 'DAY_NAME', 'WEEK_NUMBER','WEEK_YEAR', 'MONTH', 'MONTH_NAME',
    'QUARTER', 'YEAR_MONTH', 'YEAR_QUARTER', 'MONTH_PERIOD', 'YEAR']].sample(5)}")

TRANSFORMACIÓN 3: Calcular Columnas Temporales

  ✓ Tipo de DATE: datetime64[ns]
  ✓ Fechas válidas: 122,002
  ✓ Rango: 2022-01-09 00:00:00 a 2023-07-17 00:00:00

• Columnas creadas: 7

• Dimensiones totales: 122,002 filas × 27 columnas

• Utilidad por columna:
  - MONTH_NAME   → Agrupar por mes (análisis estacional)
  - DAY_WEEK     → Análisis numérico de días (0=Lunes, 6=Domingo)
  - DAY_NUMBER   → Visualizaciones legibles de días
  - QUARTER      → Análisis trimestral
  - YEAR_MONTH   → Series temporales mensuales
  - YEAR_QUARTER → Comparación entre trimestres
  - MONTH_PERIOD → Patrones de compra por quincena/pago

• Muestra de 5 filas aleatorias de fechas: 
             DATE  DAY_NUMBER DAY_NAME  WEEK_NUMBER WEEK_YEAR  MONTH  \
62907  2023-01-23           0   Monday            3     03-23      1   
747    2022-12-04           6   Sunday           48     48-22     12   
66651  2022-10-16           6   Sunday           41     41-22     10   
42538  2022-03-13           6   Sunday  

### TRANSFORMACIÓN 4: Calcular Columnas con Métricas de Ventas

In [76]:
print("="*100)
print("TRANSFORMACIÓN 4: Calcular Columnas con Métricas de Ventas")
print("="*100)

# MÉTRICA 1: PRICE - precio unitario
df_consolidado['PRICE'] = (
    df_consolidado['TOTAL_VALUE_SALES'] / df_consolidado['TOTAL_UNIT_SALES']
).replace([np.inf, -np.inf], np.nan).round(2) # Convierte infinitos a NaN (Not a Number)

print("\n" + "─"*70)
print("PRICE: precio unitario")

print(f"\n• Verificación:")
precios_validos = df_consolidado['PRICE'].notna().sum()
precios_nulos = df_consolidado['PRICE'].isna().sum()

print(f"  - Precios válidos: {precios_validos:,} ({precios_validos/len(df_consolidado)*100:.1f}%)")
print(f"  - Precios NaN: {precios_nulos:,} ({precios_nulos/len(df_consolidado)*100:.1f}%)")
print(f"  - Rango: ${df_consolidado['PRICE'].min():.2f} - ${df_consolidado['PRICE'].max():.2f}")
print(f"  - Promedio: ${df_consolidado['PRICE'].mean():.2f}")
print("─"*70 + "\n")


# MÉTRICA 2: VAR_WEEKLY_AVG - variación absoluta
df_consolidado['VAR_WEEKLY_AVG'] = (
    df_consolidado['TOTAL_UNIT_SALES'] - df_consolidado['TOTAL_UNIT_AVG_WEEKLY_SALES']
).round(2)

print("\n" + "─"*70)
print("VAR_WEEKLY_AVG: variación absoluta")

print(f"\n• Interpretación:")
inter1 = {
    'Valor positivo': 'se vendió MÁS que el promedio',
    'Valor negativo': 'se vendió MENOS que el promedio',
    'Cero': 'se vendió exactamente el promedio'
}
for col, val in inter1.items():
    print(f"   {col:15} → {val}")

print(f"\n• Verificación:")
var_positiva = (df_consolidado['VAR_WEEKLY_AVG'] > 0).sum()
var_negativa = (df_consolidado['VAR_WEEKLY_AVG'] < 0).sum()
print(f"  - Ventas sobre promedio: {var_positiva:,} ({var_positiva/len(df_consolidado)*100:.1f}%)")
print(f"  - Ventas bajo promedio: {var_negativa:,} ({var_negativa/len(df_consolidado)*100:.1f}%)")
print("─"*70 + "\n")


# MÉTRICA 3: VAR_PCT - variación porcentual
df_consolidado['VAR_PCT'] = (
    (df_consolidado['TOTAL_UNIT_SALES'] / df_consolidado['TOTAL_UNIT_AVG_WEEKLY_SALES'] - 1) * 100
).replace([np.inf, -np.inf], np.nan).round(2)

print("\n" + "─"*70)
print("VAR_PCT: variación porcentual")

print(f"\n• Interpretación:")
inter2 = {
    '+25%': 'se vendió 25% MÁS que el promedio',
    '-50%': 'se vendió 50% MENOS que el promedio',
    '0%': 'se vendió exactamente el promedio',
    '+100%': 'se vendió el DOBLE del promedio'
}
for col, val in inter2.items():
    print(f"   {col:5} → {val}")

print(f"\n• Verificación:")
print(f"  - Rango: {df_consolidado['VAR_PCT'].min():.2f}% a {df_consolidado['VAR_PCT'].max():.2f}%")
print(f"  - Promedio: {df_consolidado['VAR_PCT'].mean():.2f}%")
print("─"*70 + "\n")


# MÉTRICA 4: ABOVE_AVG - indicador binario de desempeño
df_consolidado['ABOVE_AVG'] = (
    df_consolidado['TOTAL_UNIT_SALES'] > df_consolidado['TOTAL_UNIT_AVG_WEEKLY_SALES']
).astype(int)

print("\n" + "─"*70)
print("ABOVE_AVG: indicador binario de desempeño")

print(f"\n• Interpretación:")
inter3 = {
    '1': 'la venta SUPERÓ el promedio (buen desempeño)',
    '0': 'la venta NO superó el promedio (bajo/igual al promedio)',
}
for col, val in inter3.items():
    print(f"   {col:3} → {val}")

print(f"\n• Verificación:")
ventas_sobre_promedio = df_consolidado['ABOVE_AVG'].sum()
pct_sobre_promedio = (ventas_sobre_promedio / len(df_consolidado)) * 100
print(f"  - Ventas sobre promedio (1): {ventas_sobre_promedio:,} ({pct_sobre_promedio:.1f}%)")
print(f"  - Ventas bajo promedio (0): {len(df_consolidado) - ventas_sobre_promedio:,} ({100-pct_sobre_promedio:.1f}%)")
print("─"*70 + "\n")


# RESUMEN DE TRANSFORMACIÓN 4
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 5")
print(":"*100)

print(f"\n• Columnas creadas: 4")

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'PRICE': 'precio unitario ($/unidad)',
    'VAR_WEEKLY_AVG': 'diferencia absoluta vs promedio semanal (unidades)',
    'VAR_PCT': 'diferencia relativa vs promedio semanal (%)',
    'ABOVE_AVG': 'clasificación binaria de desempeño (0/1)',
}
for col, uso in utilidad.items():
    print(f"  - {col:15} → {uso}")
    
print(f"\n• Muestra de 5 filas aleatorias de ventas: \n{df_consolidado
    [['PRICE', 'VAR_WEEKLY_AVG', 'VAR_PCT', 'ABOVE_AVG','TOTAL_UNIT_SALES', 
    'TOTAL_VALUE_SALES', 'TOTAL_UNIT_AVG_WEEKLY_SALES']].sample(5)}")

TRANSFORMACIÓN 4: Calcular Columnas con Métricas de Ventas

──────────────────────────────────────────────────────────────────────
PRICE: precio unitario

• Verificación:
  - Precios válidos: 121,924 (99.9%)
  - Precios NaN: 78 (0.1%)
  - Rango: $0.55 - $298.30
  - Promedio: $55.12
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
VAR_WEEKLY_AVG: variación absoluta

• Interpretación:
   Valor positivo  → se vendió MÁS que el promedio
   Valor negativo  → se vendió MENOS que el promedio
   Cero            → se vendió exactamente el promedio

• Verificación:
  - Ventas sobre promedio: 4,307 (3.5%)
  - Ventas bajo promedio: 117,679 (96.5%)
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
VAR_PCT: variación porcentual

• Interpretación:
   +25%  → se vendió 25% MÁS que el promedio
   -50%  → se vendió 50

### TRANSFORMACIÓN 5: Calcular Columnas para Categorizar Ventas

In [78]:
print("="*100)
print("TRANSFORMACIÓN 5: Calcular Columnas para Categorizar Ventas")
print("="*100)

# CATEGORÍA 1: CAT_SALES - por valor monetario
df_consolidado['CAT_SALES'] = pd.cut(
    df_consolidado['TOTAL_VALUE_SALES'],
    bins=[0, 10, 50, 100, 500, float('inf')],
    labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto'],
    include_lowest=True
)

print("\n" + "─"*70)
print("CAT_SALES: categoría por monto de venta ($)")

print(f"\n• Intervalos definidos:")
category1 = {
    '[0-10]': 'Muy Bajo',
    '(10-50]': 'Bajo',
    '(50-100]': 'Medio',
    '(100-500]': 'Alto',
    '(500-∞)': 'Muy Alto'
}
for col, cat in category1.items():
    print(f"   {col:10} → {cat}")

print(f"\n• Distribución:")
dist_sales = df_consolidado['CAT_SALES'].value_counts().sort_index()
for cat, count in dist_sales.items():
    pct = (count / len(df_consolidado)) * 100
    print(f"   {cat:10} : {count:6,} ventas ({pct:.1f}%)")
print("─"*70 + "\n")


# CATEGORÍA 2: CAT_UNITS - por unidades vendidas
df_consolidado['CAT_UNITS'] = pd.cut(
    df_consolidado['TOTAL_UNIT_SALES'],
    bins=[0, 0.5, 2, 5, 10, float('inf')],
    labels=['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto'],
    include_lowest=True
)

print("\n" + "─"*70)
print("CAT_UNITS: categoría por unidades vendidas")

print(f"\n• Intervalos definidos:")
category2 = {
    '[0-0.5]': 'Muy Bajo',
    '(0.5-2]': 'Bajo',
    '(2-5]': 'Medio',
    '(5-10]': 'Alto',
    '(10-∞)': 'Muy Alto'
}
for col, cat in category2.items():
    print(f"   {col:10} → {cat}")

print(f"\n• Distribución:")
dist_units = df_consolidado['CAT_UNITS'].value_counts().sort_index()
for cat, count in dist_units.items():
    pct = (count / len(df_consolidado)) * 100
    print(f"   {cat:10} : {count:6,} ventas ({pct:.1f}%)")
print("─"*70 + "\n")


# CATEGORÍA 3: CAT_PRICE - por precio unitario
df_consolidado['CAT_PRICE'] = pd.cut(
    df_consolidado['PRICE'],
    bins=[0, 20, 50, 100, 200, float('inf')],
    labels=['Económico', 'Bajo', 'Medio', 'Alto', 'Premium'],
    include_lowest=True
)

print("\n" + "─"*70)
print("CAT_PRICE: categoría por precio unitario ($/unidad)")

print(f"\n• Intervalos definidos:")
category3 = {
    '[0-20]': 'Económico',
    '(20-50]': 'Bajo',
    '(50-100]': 'Medio',
    '(100-200]': 'Alto',
    '(200-∞)': 'Premium'
}
for col, cat in category3.items():
    print(f"   {col:10} → {cat}")

print(f"\n• Distribución:")
dist_price = df_consolidado['CAT_PRICE'].value_counts().sort_index()
for cat, count in dist_price.items():
    pct = (count / len(df_consolidado)) * 100
    print(f"   {cat:10} : {count:6,} ventas ({pct:.1f}%)")
print("─"*70 + "\n")


# RESUMEN DE TRANSFORMACIÓN 5
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 5")
print(":"*100)

print(f"\n• Columnas creadas: 3")

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'CAT_SALES': 'categoría por monto de venta ($)',
    'CAT_UNITS': 'categoría por unidades vendidas',
    'CAT_PRICE': 'categoría por precio unitario ($/unidad)',
}
for col, uso in utilidad.items():
    print(f"  - {col:15} → {uso}")

print(f"\n• Muestra de 5 filas aleatorias comparativas: \n{df_consolidado
    [['TOTAL_VALUE_SALES', 'CAT_SALES','TOTAL_UNIT_SALES', 
      'CAT_UNITS','PRICE', 'CAT_PRICE']].sample(5)}")

print(f"\n• Estadísticas numéricas: \n{df_consolidado.describe()}")

TRANSFORMACIÓN 5: Calcular Columnas para Categorizar Ventas

──────────────────────────────────────────────────────────────────────
CAT_SALES: categoría por monto de venta ($)

• Intervalos definidos:
   [0-10]     → Muy Bajo
   (10-50]    → Bajo
   (50-100]   → Medio
   (100-500]  → Alto
   (500-∞)    → Muy Alto

• Distribución:
   Muy Bajo   : 50,399 ventas (41.3%)
   Bajo       : 35,943 ventas (29.5%)
   Medio      : 14,776 ventas (12.1%)
   Alto       : 16,551 ventas (13.6%)
   Muy Alto   :  4,333 ventas (3.6%)
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
CAT_UNITS: categoría por unidades vendidas

• Intervalos definidos:
   [0-0.5]    → Muy Bajo
   (0.5-2]    → Bajo
   (2-5]      → Medio
   (5-10]     → Alto
   (10-∞)     → Muy Alto

• Distribución:
   Muy Bajo   : 67,777 ventas (55.6%)
   Bajo       : 27,934 ventas (22.9%)
   Medio      : 12,051 ventas (9.9%)
   Alto       :  6,694 

### TRANSFORMACIÓN 6: Simplificar Región y Extraer Tamaño del Producto

In [80]:
print("="*100)
print("TRANSFORMACIÓN 6: Simplificar Región y Extraer Tamaño del Producto")
print("="*100)


# REGION: eliminar prefijo "Total Autos" 
df_consolidado['REGION_CLEAN'] = (
    df_consolidado['REGION']
    .str.replace('Total Autos ', '', regex=False) # Elimina prefijo repetitivo
    .str.title() # Capitaliza
)

print("\n" + "─"*70)
print(f"REGION_CLEAN: eliminar prefijo repetitivo 'Total Autos' para simplificar")
print(f"   Antes: 'Total Autos Area 1'          → Después: 'Area 1'")
print(f"   Antes: 'Total Autos Scanning Mexico' → Después: 'Scanning Mexico'")
print("─"*70 + "\n")

# REGION_SHORT: eliminar prefijo "Scanning"
df_consolidado['REGION_SHORT'] = df_consolidado['REGION_CLEAN'].str.replace('Scanning ', '', regex=False)

print("\n" + "─"*70)
print(f"REGION_SHORT: eliminar 'Scanning' para versión más corta")
print(f"   Antes:  'Scanning Mexico' → Después: 'Mexico'")
print(f"   'Area 1' se mantiene igual")
print("─"*70 + "\n")


# SIZE: extraer tamaño del producto y sus unidades de ITEM_DESCRIPTION (no cajas, piezas o botes)
df_consolidado['SIZE'] = df_consolidado['ITEM_DESCRIPTION'].str.extract( # Extrae del texto de ITEM_DESCRIPTION
    r'(\d+\.?\d*\s*(?:ML|L|LT|G|KG|GR|Gr|Grs|Kg|Lt|Lts|Ml|M))', # Expresión regular (regex)
    expand=False).str.replace(r'\s+', '', regex=True)
# Estandarizar unidades por convención a ml, l, g, kg
unidades = {
    r'(\d+\.?\d*)ML$':  r'\1ml',
    r'(\d+\.?\d*)Ml$':  r'\1ml',
    r'(\d+\.?\d*)mL$':  r'\1ml',
    r'(\d+\.?\d*)M$':   r'\1ml',
    r'(\d+\.?\d*)LT$':  r'\1l',
    r'(\d+\.?\d*)Lt$':  r'\1l',
    r'(\d+\.?\d*)LTS$': r'\1l',
    r'(\d+\.?\d*)Lts$': r'\1l',
    r'(\d+\.?\d*)L$':   r'\1l',
    r'(\d+\.?\d*)GR$':  r'\1g',
    r'(\d+\.?\d*)Gr$':  r'\1g',
    r'(\d+\.?\d*)GRS$': r'\1g',
    r'(\d+\.?\d*)Grs$': r'\1g',
    r'(\d+\.?\d*)G$':   r'\1g',
    r'(\d+\.?\d*)KG$':  r'\1kg',
    r'(\d+\.?\d*)Kg$':  r'\1kg',
    r'(\d+\.?\d*)K$':   r'\1kg',
}
for patron, reemplazo in unidades.items():
    df_consolidado['SIZE'] = df_consolidado['SIZE'].str.replace(patron, reemplazo, regex=True)


print("\n" + "─"*70)
print(f"SIZE: extracción y estandarización del tamaño del ITEM_DESCRIPTION")
print(f"   Patrón: números seguidos de unidad (ml, l, g, kg)")
print(f"   Ejemplo: 'Cloralex Bot.Plast. 500ML' → SIZE='500ML'")
print("─"*70 + "\n")


# SIZE_NUM: extraer solo el número del tamaño del producto (para cálculos)
df_consolidado['SIZE_NUM'] = pd.to_numeric(df_consolidado['SIZE'].str.extract(r'(\d+\.?\d*)', expand=False), errors='coerce')

print("\n" + "─"*70)
print(f"SIZE_NUM: número del tamaño")
print(f"   '500g' → 500.0")
print(f"   '3.75l' → 3.75")
print("─"*70 + "\n")


# SIZE_UNIT: extraer unidad de medida (para comparar entre productos)
df_consolidado['SIZE_UNIT'] = df_consolidado['SIZE'].str.extract(r'([a-z]+)$', expand=False)

print("\n" + "─"*70)
print(f"SIZE_UNIT: unidad de medida")
print(f"   '500g' → 'g'")
print(f"   '3.75l' → 'l'")
print("─"*70 + "\n")



# RESUMEN DE TRANSFORMACIÓN 6
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 6")
print(":"*100)

print(f"\n• Columnas creadas: 5")

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'REGION_CLEAN': 'región simplificada (sin Total Autos)',
    'REGION_SHORT': 'región corta (sin Total Autos Scanning)',
    'SIZE': 'tamaño del producto con número y unidades',
    'SIZE_NUM': 'número del tamaño del producto',
    'SIZE_UNIT': 'unidades del tamaño del producto'
}
for col, uso in utilidad.items():
    print(f"  - {col:15} → {uso}")

print(f"\n• Nulos en SIZE: {df_consolidado['SIZE'].isnull().sum():,} ({df_consolidado['SIZE'].isnull().mean()*100:.1f}%)")
print(f"   Productos que no tienen patrón de tamaño estándar (como botes o cajas)")

print(f"\n• Unidades:")
print(df_consolidado['SIZE_UNIT'].value_counts())

print(f"\n• Muestra de 5 filas aleatorias: \n{df_consolidado
    [['REGION_CLEAN', 'REGION_SHORT', 'ITEM_DESCRIPTION', 
      'SIZE', 'SIZE_NUM', 'SIZE_UNIT']].sample(5)}")

TRANSFORMACIÓN 6: Simplificar Región y Extraer Tamaño del Producto

──────────────────────────────────────────────────────────────────────
REGION_CLEAN: eliminar prefijo repetitivo 'Total Autos' para simplificar
   Antes: 'Total Autos Area 1'          → Después: 'Area 1'
   Antes: 'Total Autos Scanning Mexico' → Después: 'Scanning Mexico'
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
REGION_SHORT: eliminar 'Scanning' para versión más corta
   Antes:  'Scanning Mexico' → Después: 'Mexico'
   'Area 1' se mantiene igual
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
SIZE: extracción y estandarización del tamaño del ITEM_DESCRIPTION
   Patrón: números seguidos de unidad (ml, l, g, kg)
   Ejemplo: 'Cloralex Bot.Plast. 500ML' → SIZE='500ML'
────────────────────────────────────────────────────────────

### TRANSFORMACIÓN 7: Calcular Columnas con Indicadores de Negocio

In [81]:
print("="*100)
print("TRANSFORMACIÓN 7: Calcular Columnas con Indicadores de Negocio")
print("="*100)


# CALCULAR MEDIANAS PARA CLASIFICACIÓN
mediana_precio = df_consolidado['PRICE'].median()
mediana_rotacion = df_consolidado['TOTAL_UNIT_AVG_WEEKLY_SALES'].median()
mediana_valor = df_consolidado['TOTAL_VALUE_SALES'].median()
    # La mediana es más robusta ante valores extremos (outliers)

print(f"""\n
MEDIANAS CALCULADAS:
  • Precio (PRICE):              ${mediana_precio:.2f}
  • Rotación (AVG_WEEKLY_SALES): {mediana_rotacion:.3f} unidades
  • Valor (TOTAL_VALUE_SALES):   ${mediana_valor:.2f}
""")


# INDICADOR 1: HIGH_VALUE - producto de alto valor
df_consolidado['HIGH_VALUE'] = (
    df_consolidado['PRICE'] > mediana_precio
).astype(int)

print("\n" + "─"*70)
print(f"HIGH_VALUE: precio por encima de la mediana (${mediana_precio:.2f})")
print(f"\n• Interpretación:")
print(f"   0 = precio ≤ ${mediana_precio:.2f}")
print(f"   1 = precio > ${mediana_precio:.2f}")

print(f"\n• Verificación:")
pct_high = df_consolidado['HIGH_VALUE'].mean() * 100
print(f"   {pct_high:.1f}% de ventas son de alto valor")
print("─"*70 + "\n")


# INDICADOR 2: HIGH_TURNOVER - alta rotación
df_consolidado['HIGH_TURNOVER'] = (
    df_consolidado['TOTAL_UNIT_AVG_WEEKLY_SALES'] > mediana_rotacion
).astype(int)

print("\n" + "─"*70)
print(f"HIGH_TURNOVER: promedio de ventas semanal por encima de la mediana ({mediana_rotacion:.3f} uds)")
print(f"\n• Interpretación:")
print(f"   0 = rotación ≤ {mediana_rotacion:.3f} unidades/semana")
print(f"   1 = rotación > {mediana_rotacion:.3f} unidades/semana")

print(f"\n• Verificación:")
pct_turnover = df_consolidado['HIGH_TURNOVER'].mean() * 100
print(f"   {pct_turnover:.1f}% de ventas son de alta rotación")
print("─"*70 + "\n")


# INDICADOR 3: VIP_SALE - ventas de mayor impacto
df_consolidado['VIP_SALE'] = (
    df_consolidado['TOTAL_VALUE_SALES'] > mediana_valor
).astype(int)

print("\n" + "─"*70)
print(f"VIP_SALE: valor de venta por encima de la mediana (${mediana_valor:.2f})")
print(f"\n• Interpretación:")
print(f"   0 = venta ≤ ${mediana_valor:.2f}")
print(f"   1 = venta > ${mediana_valor:.2f}")

print(f"\n• Verificación:")
pct_vip = df_consolidado['VIP_SALE'].mean() * 100
print(f"   {pct_vip:.1f}% de ventas son de mayor impacto")
print("─"*70 + "\n")


# INDICADOR 4: STAR_PRODUCT - producto estrella con alto valor y alta rotación
df_consolidado['STAR_PRODUCT'] = (
    (df_consolidado['HIGH_VALUE'] == 1) & 
    (df_consolidado['HIGH_TURNOVER'] == 1)
).astype(int)

print("\n" + "─"*70)
print(f"STAR_PRODUCT: producto con HIGH_VALUE=1 Y HIGH_TURNOVER=1")
print(f"\n• Interpretación:")
print(f"   0 = no cumple ambas condiciones")
print(f"   1 = precio alto (>${mediana_precio:.2f}) y alta rotación (>{mediana_rotacion:.3f} uds)")

print(f"\n• Verificación:")
pct_star = df_consolidado['STAR_PRODUCT'].mean() * 100
n_star = df_consolidado[df_consolidado['STAR_PRODUCT'] == 1]['ITEM_CODE'].nunique()
print(f"   {pct_star:.1f}% de ventas son de alto valor y alta rotación")
print(f"   Productos estrella únicos: {n_star}")
print("─"*70 + "\n")



# RESUMEN DE TRANSFORMACIÓN 7
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 7")
print(":"*100)

print(f"\n• Columnas creadas: 4")

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'HIGH_VALUE': 'producto de precio por encima de la mitad del portafolio',
    'HIGH_TURNOVER': 'producto que se vende en mayor volumen que la mitad del portafolio',
    'VIP_SALE': 'venta que supera la mitad de las transacciones',
    'STAR_PRODUCT': 'producto con precio alto y alta demanda simultáneamente'
}
for col, uso in utilidad.items():
    print(f"  - {col:15} → {uso}")

print(f"\n• Muestra de 5 filas aleatorias comparativas: \n{df_consolidado
    [['PRICE', 'TOTAL_UNIT_AVG_WEEKLY_SALES','TOTAL_VALUE_SALES', 
      'HIGH_VALUE', 'HIGH_TURNOVER', 'VIP_SALE', 'STAR_PRODUCT']].sample(5)}")

TRANSFORMACIÓN 7: Calcular Columnas con Indicadores de Negocio


MEDIANAS CALCULADAS:
  • Precio (PRICE):              $36.64
  • Rotación (AVG_WEEKLY_SALES): 3.994 unidades
  • Valor (TOTAL_VALUE_SALES):   $16.81


──────────────────────────────────────────────────────────────────────
HIGH_VALUE: precio por encima de la mediana ($36.64)

• Interpretación:
   0 = precio ≤ $36.64
   1 = precio > $36.64

• Verificación:
   50.0% de ventas son de alto valor
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
HIGH_TURNOVER: promedio de ventas semanal por encima de la mediana (3.994 uds)

• Interpretación:
   0 = rotación ≤ 3.994 unidades/semana
   1 = rotación > 3.994 unidades/semana

• Verificación:
   50.0% de ventas son de alta rotación
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
VIP_SALE: valor de

### TRANSFORMACIÓN 8: Calcular Columnas de Ranking y Percentiles

In [82]:
print("="*100)
print("TRANSFORMACIÓN 8: Calcular Columnas de Ranking y Percentiles")
print("="*100)

# RANKING 1: RANK_PRODUCT_VALUE - ranking global de productos
df_consolidado['RANK_PRODUCT_VALUE'] = df_consolidado.groupby('ITEM_CODE')['TOTAL_VALUE_SALES'].transform('sum').rank(ascending=False, method='dense')
    # groupby('ITEM_CODE')['TOTAL_VALUE_SALES']: agrupa ventas por producto
    # .transform('sum'): suma ventas totales por producto (en cada fila)
    # .rank(ascending=False,: asignar rank (1 = más vendido)
    # method='dense'): sin saltos en el ranking (1,2,2,3 no 1,2,2,4)

print("\n" + "─"*70)
print(f"RANK_PRODUCT_VALUE: posición del producto por ventas totales históricas")
print(f"\n• Interpretación:")
print(f"   1 es el producto más vendido")
print(f"   343 es el producto menos vendido")

print(f"\n• Verificación:")
print(f"   Rango: {int(df_consolidado['RANK_PRODUCT_VALUE'].min())} - {int(df_consolidado['RANK_PRODUCT_VALUE'].max())}")
print(f"   Valores únicos: {df_consolidado['RANK_PRODUCT_VALUE'].nunique()}")
print("─"*70 + "\n")


# RANKING 2: PERCENTILE_REGION - percentil dentro de la región
df_consolidado['PERCENTILE_REGION'] = df_consolidado.groupby('REGION')['TOTAL_VALUE_SALES'].transform(
    lambda x: pd.qcut(x, q=10, labels=False, duplicates='drop'))
    # groupby('REGION')['TOTAL_VALUE_SALES']: agrupa ventas por región
    # pd.qcut(q=10): divide en 10 grupos iguales (deciles)
    # labels=False → devuelve 0-9 en lugar de etiquetas
    # duplicates='drop': maneja valores duplicados en los límites

print("\n" + "─"*70)
print(f"PERCENTILE_REGION: posición de la venta dentro de su región (0-9)")
print(f"\n• Interpretación:")
print(f"   0 = 10% inferior de su región")
print(f"   9 = 10% superior de su región")

print(f"\n• Verificación:")
print(f"    Distribución de percentiles:")
dist_pct = df_consolidado['PERCENTILE_REGION'].value_counts().sort_index()
for pct, count in dist_pct.items():
    print(f"     - Percentil {pct}: {count:,} ventas")
print("─"*70 + "\n")


# RANKING 3: RANK_YEAR_ITEM - ranking de ventas individuales por producto/año
df_consolidado['RANK_YEAR_ITEM'] = df_consolidado.groupby(['YEAR', 'ITEM_CODE'])['TOTAL_VALUE_SALES'].rank(ascending=False)
    # groupby(['YEAR', 'ITEM_CODE'])['TOTAL_VALUE_SALES']: agrupa ventas por año + producto
    # .rank(ascending=False): ordena ventas de mayor a menor, cada venta (semana+región) recibe su posición dentro del grupo

print("\n" + "─"*70)
print(f"RANK_YEAR_ITEM: posición de cada venta individual del producto en ese año")
print(f"\n• Interpretación:")
print(f"   Rank 1 = la venta más alta")
print(f"   Rank 364 = la venta más baja")

print(f"\n• Verificación:")
print(f"   Rango: {int(df_consolidado['RANK_YEAR_ITEM'].min())} - {int(df_consolidado['RANK_YEAR_ITEM'].max())}")
print("─"*70 + "\n")



# RESUMEN DE TRANSFORMACIÓN 8
print(f"\n" + ":"*100)
print("RESUMEN DE TRANSFORMACIÓN 8")
print(":"*100)

print(f"\n• Columnas creadas: 3")

print(f"\n• Dimensiones totales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

print(f"\n• Utilidad por columna:")
utilidad = {
    'RANK_PRODUCT_VALUE': 'posición del producto por ventas totales históricas (1 = item más vendido)',
    'PERCENTILE_REGION': 'posición de la venta dentro de su región (0-9)',
    'RANK_YEAR_ITEM': 'posición de cada venta individual dentro del producto en ese año (1 = venta más alta del item)'
}
for col, uso in utilidad.items():
    print(f"  - {col:15} → {uso}")

print(f"\n• Muestra de 5 filas aleatorias con contexto: \n{df_consolidado
    [['ITEM_CODE', 'WEEK_YEAR', 'REGION_SHORT', 'TOTAL_VALUE_SALES',
    'RANK_PRODUCT_VALUE', 'PERCENTILE_REGION', 'RANK_YEAR_ITEM']].sample(5)}")

TRANSFORMACIÓN 8: Calcular Columnas de Ranking y Percentiles

──────────────────────────────────────────────────────────────────────
RANK_PRODUCT_VALUE: posición del producto por ventas totales históricas

• Interpretación:
   1 es el producto más vendido
   343 es el producto menos vendido

• Verificación:
   Rango: 1 - 343
   Valores únicos: 343
──────────────────────────────────────────────────────────────────────


──────────────────────────────────────────────────────────────────────
PERCENTILE_REGION: posición de la venta dentro de su región (0-9)

• Interpretación:
   0 = 10% inferior de su región
   9 = 10% superior de su región

• Verificación:
    Distribución de percentiles:
     - Percentil 0: 12,226 ventas
     - Percentil 1: 12,185 ventas
     - Percentil 2: 12,193 ventas
     - Percentil 3: 12,198 ventas
     - Percentil 4: 12,202 ventas
     - Percentil 5: 12,198 ventas
     - Percentil 6: 12,199 ventas
     - Percentil 7: 12,202 ventas
     - Percentil 8: 12,196 ventas

### TRANSFORMACIÓN 9: Reordenar Columnas

In [85]:
print("="*100)
print("TRANSFORMACIÓN 9: Reordenar Columnas")
print("="*100)

# Definir orden lógico
columnas_ordenadas = [
    
    # Identificadores del producto
    'ITEM_CODE',
    'MANUFACTURER',
    'BRAND',
    'ITEM_DESCRIPTION',
    
    # Categorización del producto
    'ID_CATEGORY',
    'CATEGORY',
    'SEGMENT',
    'FORMAT',
    'ATTR1',
    'ATTR2',
    'ATTR3',
    
    # Tamaño del producto
    'SIZE',
    'SIZE_NUM',
    'SIZE_UNIT',
    
    # Temporal
    'DATE', 
    'DAY_NUMBER', 
    'DAY_NAME', 
    'WEEK_NUMBER',
    'WEEK_YEAR', 
    'MONTH', 
    'MONTH_NAME',
    'QUARTER', 
    'YEAR_MONTH', 
    'YEAR_QUARTER', 
    'MONTH_PERIOD', 
    'YEAR',
    
    # Ubicación
    'REGION',
    'REGION_CLEAN',
    'REGION_SHORT',
    
    # Métricas de ventas
    'TOTAL_UNIT_SALES',   
    'TOTAL_VALUE_SALES',      
    'TOTAL_UNIT_AVG_WEEKLY_SALES',
    'PRICE',
    'VAR_WEEKLY_AVG',
    'VAR_PCT',
    
    # Categorizaciones
    'CAT_SALES',
    'CAT_UNITS',
    'CAT_PRICE',
    
    # Indicadores binarios
    'ABOVE_AVG',
    'HIGH_VALUE',
    'HIGH_TURNOVER',
    'VIP_SALE',
    'STAR_PRODUCT',
    
    # Rankings y percentiles
    'RANK_PRODUCT_VALUE',
    'PERCENTILE_REGION',
    'RANK_YEAR_ITEM'
]

df_consolidado = df_consolidado[columnas_ordenadas]

print(f"• Columnas reorganizadas en 9 grupos:")
print(f"\n{'Grupo':<35} {'Columnas':<10}")
print("─"*50)
grupos = {
    'Identificadores del producto': '4  (1 - 4)',
    'Categorización del producto': '7  (5 - 11)',
    'Tamaño del producto': '3  (12 - 14)',
    'Temporal': '12 (15 - 26)',
    'Ubicación': '3  (27 - 29)',
    'Métricas de ventas': '6  (30 - 35)',
    'Categorizaciones': '3  (36 - 38)',
    'Indicadores binarios': '5  (39 - 43)',
    'Rankings y percentiles': '3  (44 - 46)'
}
for grupo, cols in grupos.items():
    print(f" {grupo:30} →   {cols}")

print("\n• Listado de columnas finales:")
for i, col in enumerate(df_consolidado.columns, 1):
    print(f"{i:2}.{col}")

print(f"\n• Dimensiones finales: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")

TRANSFORMACIÓN 9: Reordenar Columnas
• Columnas reorganizadas en 9 grupos:

Grupo                               Columnas  
──────────────────────────────────────────────────
 Identificadores del producto   →   4  (1 - 4)
 Categorización del producto    →   7  (5 - 11)
 Tamaño del producto            →   3  (12 - 14)
 Temporal                       →   12 (15 - 26)
 Ubicación                      →   3  (27 - 29)
 Métricas de ventas             →   6  (30 - 35)
 Categorizaciones               →   3  (36 - 38)
 Indicadores binarios           →   5  (39 - 43)
 Rankings y percentiles         →   3  (44 - 46)

• Listado de columnas finales:
 1.ITEM_CODE
 2.MANUFACTURER
 3.BRAND
 4.ITEM_DESCRIPTION
 5.ID_CATEGORY
 6.CATEGORY
 7.SEGMENT
 8.FORMAT
 9.ATTR1
10.ATTR2
11.ATTR3
12.SIZE
13.SIZE_NUM
14.SIZE_UNIT
15.DATE
16.DAY_NUMBER
17.DAY_NAME
18.WEEK_NUMBER
19.WEEK_YEAR
20.MONTH
21.MONTH_NAME
22.QUARTER
23.YEAR_MONTH
24.YEAR_QUARTER
25.MONTH_PERIOD
26.YEAR
27.REGION
28.REGION_CLEAN
29.REGION_SHOR

### REVISÓN DE TRANSFORMACIONES

In [88]:
print("="*100)
print("REVISIÓN DE TRANSFORMACIONES")
print("="*100)

print(f"\nComprobar que no quedaron errores de tipo, nulos o duplicados")

# Verifica tipos
print(f"\n• Tipos de datos por columna: \n{df_consolidado.dtypes}")

# Verificar duplicados
print(f"\n• Filas duplicadas: {df_consolidado.duplicated().sum()}")

# Verificar nulos
nulos_totales = df_consolidado.isnull().sum().sum()
print(f"\n• Nulos totales: {nulos_totales:,}")

if nulos_totales > 0:
    print("  - Columnas con nulos por tratar:")
    nulos_col = df_consolidado.isnull().sum()
    for col, count in nulos_col[nulos_col > 0].items():
        pct = (count / len(df_consolidado)) * 100
        print(f"    {col:11}:  {count:2,} ({pct:.2f}%)")

print(f"\n• Dimensiones finales: {df_consolidado.shape[0]:,} filas x {df_consolidado.shape[1]} columnas")

REVISIÓN DE TRANSFORMACIONES

Comprobar que no quedaron errores de tipo, nulos o duplicados

• Tipos de datos por columna: 
ITEM_CODE                              object
MANUFACTURER                           object
BRAND                                  object
ITEM_DESCRIPTION                       object
ID_CATEGORY                             int64
CATEGORY                               object
SEGMENT                                object
FORMAT                                 object
ATTR1                                  object
ATTR2                                  object
ATTR3                                  object
SIZE                                   object
SIZE_NUM                              float64
SIZE_UNIT                              object
DATE                           datetime64[ns]
DAY_NUMBER                              int32
DAY_NAME                               object
WEEK_NUMBER                             int64
WEEK_YEAR                              object
MO

### TRATAR NULOS FINALES

In [92]:
print("="*100)
print("TRATAR NULOS FINALES")
print("="*100)

# SIZE, SIZE_NUM, SIZE_UNIT

# Si ITEM_DESCRIPTION lo contiene, rellenar con '3.785l'
mascara_3785 = df_consolidado['SIZE'].isnull() & df_consolidado['ITEM_DESCRIPTION'].str.contains('3.785', na=False)
df_consolidado.loc[mascara_3785, 'SIZE'] = '3.785l'
df_consolidado.loc[mascara_3785, 'SIZE_NUM'] = 3.785
df_consolidado.loc[mascara_3785, 'SIZE_UNIT'] = 'l'

# Sin tamaño reconocible rellenar con 'Sin Especificar'
mascara_resto = df_consolidado['SIZE'].isnull()
df_consolidado.loc[mascara_resto, 'SIZE'] = 'Sin Especificar'
df_consolidado.loc[mascara_resto, 'SIZE_NUM'] = 0
df_consolidado.loc[mascara_resto, 'SIZE_UNIT'] = 'Sin Especificar'

print(f"""\n• Nulos de SIZE: 
   - Rellenados con '3.785l': {mascara_3785.sum()} filas
   - Rellenados con 'Sin Especificar': {mascara_resto.sum()} filas""")

print(f"""\n• Nulos de SIZE_NUM: 
   - Rellenados con '3.785': {mascara_3785.sum()} filas
   - Rellenados con '0': {mascara_resto.sum()} filas""")

print(f"""\n• Nulos de SIZE_UNIT: 
   - Rellenados con 'l': {mascara_3785.sum()} filas
   - Rellenados con 'Sin Especificar': {mascara_resto.sum()} filas""")


# PRICE

# Rellenar con 0 porque TOTAL_UNIT_SALES = 0
df_consolidado['PRICE'] = df_consolidado['PRICE'].fillna(0)

print(f"""\n• Nulos de PRICE:
   - Rellenados con 0: 78 filas""")


# CATEGORY_PRICE

# Recrear categorías sin incluir precios = 0 (NaN)
df_consolidado['CAT_PRICE'] = pd.cut(
    df_consolidado['PRICE'],
    bins=[0, 20, 50, 100, 200, float('inf')],
    labels=['Económico', 'Bajo', 'Medio', 'Alto', 'Premium'],
    include_lowest=False
)
# Agregar nueva categoría 'Sin Especificar'
df_consolidado['CAT_PRICE'] = df_consolidado['CAT_PRICE'].cat.add_categories(['Sin Especificar'])

# Asignar 'Sin Especificar' a precios = 0
mascara_cero = df_consolidado['PRICE'] == 0
df_consolidado.loc[mascara_cero, 'CAT_PRICE'] = 'Sin Especificar'

print(f"""\n• Nulos de CAT_PRICE:
   - Categoría 'Sin Especificar' creada para precio = 0""")



# VERIFICACIÓN FINAL
print(f"\n" + ":"*100)
print("VERIFICACIÓN FINAL")
print(":"*100)

nulos_totales = df_consolidado.isnull().sum().sum()
print(f"\n• Nulos totales: {nulos_totales:,}")

print("\n• Distribución de CAT_PRICE:")
print(df_consolidado['CAT_PRICE'].value_counts().sort_index())

print("\n• 5 ejemplos aleatorios de PRICE = 0 y CAT_PRICE = Sin Especificar:")
print(df_consolidado[df_consolidado['PRICE'] == 0][['TOTAL_UNIT_SALES', 'PRICE', 'CAT_PRICE']].sample(5))

print("\n• 5 ejemplos aleatorios de SIZE = Sin Especificar:")
print(df_consolidado[df_consolidado['SIZE'] == 'Sin Especificar'][['ITEM_DESCRIPTION', 'SIZE', 'SIZE_NUM', 'SIZE_UNIT']].sample(5))

print("\n• 5 ejemplos aleatorios de SIZE = 3.785l:")
print(df_consolidado[df_consolidado['SIZE'] == '3.785l'][['ITEM_DESCRIPTION', 'SIZE', 'SIZE_NUM', 'SIZE_UNIT']].sample(5))

TRATAR NULOS FINALES

• Nulos de SIZE: 
   - Rellenados con '3.785l': 0 filas
   - Rellenados con 'Sin Especificar': 0 filas

• Nulos de SIZE_NUM: 
   - Rellenados con '3.785': 0 filas
   - Rellenados con '0': 0 filas

• Nulos de SIZE_UNIT: 
   - Rellenados con 'l': 0 filas
   - Rellenados con 'Sin Especificar': 0 filas

• Nulos de PRICE:
   - Rellenados con 0: 78 filas

• Nulos de CAT_PRICE:
   - Categoría 'Sin Especificar' creada para precio = 0

::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
VERIFICACIÓN FINAL
::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::

• Nulos totales: 0

• Distribución de CAT_PRICE:
CAT_PRICE
Económico          27764
Bajo               47420
Medio              28360
Alto               15313
Premium             3067
Sin Especificar       78
Name: count, dtype: int64

• 5 ejemplos aleatorios de PRICE = 0 y CAT_PRICE = Sin Especificar:
       TOTAL_UNIT_SAL

## INFO DE DATAFRAME CONSOLIDADO

In [94]:
print("="*100)
print("INFO DE DATAFRAME CONSOLIDADO")
print("="*100)

print(df_consolidado.info())

INFO DE DATAFRAME CONSOLIDADO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122002 entries, 0 to 122001
Data columns (total 46 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ITEM_CODE                    122002 non-null  object        
 1   MANUFACTURER                 122002 non-null  object        
 2   BRAND                        122002 non-null  object        
 3   ITEM_DESCRIPTION             122002 non-null  object        
 4   ID_CATEGORY                  122002 non-null  int64         
 5   CATEGORY                     122002 non-null  object        
 6   SEGMENT                      122002 non-null  object        
 7   FORMAT                       122002 non-null  object        
 8   ATTR1                        122002 non-null  object        
 9   ATTR2                        122002 non-null  object        
 10  ATTR3                        122002 non-null  object        
 

## Guardar el conjunto de datos consolidado:

In [96]:
# CSV
df_consolidado.to_csv('df_consolidado_final.csv', index=False, encoding='utf-8-sig')
print("df_consolidado_final.csv")

# Excel con múltiples hojas
with pd.ExcelWriter('df_consolidado_final.xlsx', engine='openpyxl') as writer:
    # Datos completos
    df_consolidado.to_excel(writer, sheet_name='Datos Consolidados', index=False)
    # Resumen estadístico
    df_consolidado.describe(include='all').T.to_excel(writer, sheet_name='Estadísticas')
    # Info de columnas
    info_cols = pd.DataFrame({
        'Columna': df_consolidado.columns,
        'Tipo': df_consolidado.dtypes.values,
        'Nulos': df_consolidado.isnull().sum().values,
        'Únicos': df_consolidado.nunique().values
    })
    info_cols.to_excel(writer, sheet_name='Info Columnas', index=False)
print("df_consolidado_final.xlsx")

# Parquet
df_consolidado.to_parquet('df_consolidado_final.parquet', index=False)
print("df_consolidado_final.parquet")

df_consolidado_final.csv
df_consolidado_final.xlsx
df_consolidado_final.parquet
